Author : Ong Gim Shin
# Create Embeddings from the Primary Diagnosis text
Creates embeddings from publicly available pre-trained BioClinicalBert embeddings https://arxiv.org/abs/1904.03323

In [2]:
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.feature_selection import SelectKBest, SelectPercentile, chi2
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, mean_absolute_error, mean_absolute_percentage_error
import torch
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import os

output_filename = "procedures_embeddings.parquet"
output_folder = 'Merged_Dataset/'
output_path = os.path.join(output_folder, output_filename)

## 1. Read training file

In [3]:
df_train = pd.read_csv(output_folder+ 'pivot/train_los.csv')
df_train.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime',
       'first_careunit', 'outtime', 'remaining los', 'gender', 'age',
       'anchor_year', 'anchor_year_group', 'admission_type',
       'admission_location', 'insurance', 'language', 'marital_status', 'race',
       'icu_types', 'assigned_intime', 'assigned_outtime', 'num_chart_events',
       'Cume Mean ART BP Mean', 'Cume Mean Arterial Blood Pressure mean',
       'Cume Mean Heart Rate', 'Cume Mean Heart rate Alarm - High',
       'Cume Mean Height', 'Cume Mean High risk (>51) interventions',
       'Cume Mean Inspired O2 Fraction',
       'Cume Mean Low risk (25-50) interventions',
       'Cume Mean Non Invasive Blood Pressure mean',
       'Cume Mean Resp Alarm - High', 'Cume Mean Resp Alarm - Low',
       'Cume Mean Respiratory Rate', 'Cume Mean Temperature',
       'Cume Mean Weight', 'Cume Mean if_Access Lines - Invasive',
       'Cume Mean if_Cardiovascular', 'Cume Mean if_Dialysis',
       'Cume Mean if_IABP',

## 2. Load the model and tokenizer from hgf

In [4]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"
max_length=512
batch_size=16

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Check for Mac/Windows compatibility
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

## 3. Generate the embeddings

In [5]:
# Training on Mac machine
def get_bert_embeddings(texts, model, tokenizer, batch_size=16, max_length=512):

    embeddings = []
    # Check that MPS is available
    if not torch.backends.mps.is_available():
        if not torch.backends.mps.is_built():
            print("MPS not available because the current PyTorch install was not "
                  "built with MPS enabled.")
        else:
            print("MPS not available because the current MacOS version is not 12.3+ "
                  "and/or you do not have an MPS-enabled device on this machine.")
    
    else:
        device = torch.device("mps")
        model.to(device)  
        # Generates BERT embeddings for a list of texts
        model.eval()
        with torch.no_grad():
            for text in texts:
                inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)
                inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to MPS
                outputs = model(**inputs)
                # Use the cls token embedding as the sentence embedding
                cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # Move to CPU for numpy
                embeddings.append(cls_embedding)
 
    return embeddings


In [6]:
df_proc = df_train[['subject_id','hadm_id','stay_id','procedure_concat']].drop_duplicates()
df_proc['procedure_concat'] = df_proc['procedure_concat'].fillna('')

In [7]:
df_proc['procedures_embeddings'] = get_bert_embeddings(df_proc['procedure_concat'], model, tokenizer)

## 4. Save the embeddings as a parquet file

In [16]:
df_proc[['subject_id','hadm_id','stay_id','procedure_concat','procedures_embeddings']].to_parquet(output_path, index=False)
print(f"Procedure embeddings saved to {output_path}")

Procedure embeddings saved to Merged_Dataset/procedures_embeddings.parquet
